In [2]:
import subprocess, sys, torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"Python: {sys.version}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Core dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "miditok>=3.0.0",
    "ncps>=0.0.7",
    "safetensors>=0.4.0",
    "pretty_midi>=0.2.10",
    "numpy>=1.24.0",
    "tqdm>=4.65.0",
], check=False)

# Pre-built Mamba wheels — no compilation
CAUSAL_CONV1D = "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1/causal_conv1d-1.6.1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
MAMBA_SSM = "https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

if torch.cuda.is_available():
    print("\nInstalling Mamba from pre-built wheels...")
    r1 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", CAUSAL_CONV1D], capture_output=True)
    r2 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", MAMBA_SSM], capture_output=True)
    if r2.returncode != 0:
        print(f"Mamba install failed: {r2.stderr.decode()[-300:]}")
    else:
        print("Mamba wheels installed successfully")
else:
    print("No GPU — GRU fallback will be used")

# Verify
try:
    import mamba_ssm
    print(f"mamba-ssm {mamba_ssm.__version__} active — real Mamba enabled")
except ImportError:
    print("mamba-ssm not available — GRU fallback active")

sys.path.insert(0, '/kaggle/input/piano-midi-model')
print("\nSetup complete.")

PyTorch: 2.9.0+cu126
CUDA: 12.6
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
GPU: Tesla P100-PCIE-16GB

Installing Mamba from pre-built wheels...
Mamba wheels installed successfully
mamba-ssm 2.3.1 active — real Mamba enabled

Setup complete.


In [4]:
# ============================================
# EDIT THIS CELL TO CHANGE TRAINING SETTINGS
# ============================================
SCALE = "nano"        # Options: nano, micro, small, medium
MAX_EPOCHS = 2000     # Total target epochs across all sessions
# ============================================

# Show available presets
import sys
sys.path.insert(0, '/kaggle/input/datasets/chickaboomcmurtrie/magic-midi')  # update slug if needed
from scale_config import list_presets, get_preset
list_presets()
print(f"\nSelected: {SCALE}")
get_preset(SCALE)


Scale     Params    Description
------    ------    -----------
nano      ~3M       ~3M params - 30min Colab session, pipeline validation
micro     ~8M       ~8M params - meaningful learning, ~2 epochs per 30min session
small     ~22M      ~22M params - full architecture, ~1 epoch per 30min session
medium    ~60M      ~60M params - serious model, needs Colab Pro or Kaggle

Selected: nano
[nano] ~3M params - 30min Colab session, pipeline validation


{'description': '~3M params - 30min Colab session, pipeline validation',
 'params': '~3M',
 'model': ModelConfig(vocab_size=2000, d_model=192, n_layers=4, d_state=16, d_conv=4, expand=2, cfc_units=192, cfc_backbone_units=96, cfc_backbone_layers=2, cfc_every_n_layers=2, num_attention_heads=4, attention_every_n_layers=2, attention_dropout=0.1, use_relative_attention=True, max_relative_distance=128, dropout=0.1, use_cfc=True, use_mamba=True, max_sequence_length=1024, debug=False),
 'train': TrainConfig(batch_size=4, grad_accumulation_steps=4, learning_rate=0.0005, weight_decay=0.01, max_epochs=50, warmup_steps=200, max_grad_norm=1.0, save_every_n_epochs=1, keep_every_n_epochs=10, checkpoint_dir='checkpoints/', use_wandb=False, seed=42, device='auto'),
 'data': DataConfig(maestro_path='maestro-v3.0.0', tokenizer_path='tokenizer.json', processed_path='processed/', vocab_size=2000, seed_length=128, continuation_length=256, stride=128, min_piece_length=1200, max_sequence_length=512, max_piece

In [7]:
import os
# Find where your dataset is actually mounted
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'kaggle_session.py':
            print(os.path.join(root, f))

In [8]:
# Run this to check actual parameter counts on this GPU runtime
from kaggle_session import calibrate_on_kaggle
calibrate_on_kaggle()


ImportError: cannot import name 'calibrate_on_kaggle' from 'kaggle_session' (/usr/local/lib/python3.12/dist-packages/kaggle_session.py)

In [ ]:
# Generate a continuation from the best checkpoint
from kaggle_session import generate_from_best_checkpoint
generate_from_best_checkpoint(scale=SCALE)


## Downloading your trained model

After training, download your checkpoints from the Kaggle output panel:
1. Click the three dots (...) next to the notebook
2. Select "Output"
3. Download `checkpoints/best.safetensors` - this is your best model
4. Download `checkpoints/latest.safetensors` - this is the most recent epoch

To continue training in a future session:
- Re-run this notebook
- It will automatically detect and resume from the latest checkpoint
- Checkpoints persist as long as you save the notebook version

To use your model locally:
```bash
python tools/generate_sample.py \
  --checkpoint path/to/best.safetensors \
  --seed your_seed.mid \
  --output generated.mid \
  --scale nano
```
